# CSV - Duplicates

Add exact or conflicting-key duplicates, scattered or in repeated batches.

**Instructions:** Use `prepare_data_csv_update.ipynb` to generate a clean `data/weather.csv` first. The notebook overwrites it; restore clean data before rerunning.


In [ ]:
from pathlib import Path
from random import Random
import pandas as pd

# Configuration
DATA_PATH = Path("../../data/weather.csv")
TIME_COLUMN = "time"
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive; local calendar dates.
SEED = 814205

# Duplicates
KEY_COLUMNS = ["time"]
DUPLICATE_CONFLICT_COLUMN = "temperature_2m"
DUPLICATE_CONFLICT_OFFSET = 1.5
DUPLICATE_KIND = "mixed"  # "exact", "conflicting", "mixed", "random".
DUPLICATE_PLACEMENT = "mixed"  # "scattered", "grouped", "mixed", "random".
DUPLICATE_RATE = (0.01, 0.04)  # Share of eligible original rows copied once.
DUPLICATE_GROUPS = 3
DUPLICATE_GROUPED_SHARE = 0.70
DUPLICATE_CONFLICT_SHARE = 0.50


In [ ]:
# Shared placement helper: copy once into a combined dirty_data.py.
def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen

In [ ]:
def add_duplicates(
    df, rng, *, time_column, timezone, start_date, end_date,
    key_columns, conflict_column, conflict_offset, duplicate_kind,
    placement, duplicate_rate, n_groups, grouped_share, conflict_share,
):
    # Keys determine whether records are truly duplicates for this dataset.
    required = [time_column, *key_columns]
    if any(column not in df for column in required):
        raise ValueError("Time or key column not found.")

    clean = df.reset_index(drop=True).copy()
    if clean[key_columns].isna().any().any() or clean.duplicated(subset=key_columns).any():
        raise ValueError("Expected unique, nonmissing keys. Restore clean data first.")
    time = pd.to_datetime(clean[time_column], utc=True)
    if time.isna().any() or not time.is_monotonic_increasing:
        raise ValueError("Expected valid, chronologically ordered timestamps.")

    start = pd.Timestamp(start_date, tz=timezone).tz_convert("UTC")
    end = pd.Timestamp(end_date, tz=timezone).tz_convert("UTC")
    if start >= end:
        raise ValueError("Start date must precede end date.")
    scope = time.ge(start) & time.lt(end)
    sort_columns = [time_column] + [col for col in key_columns if col != time_column]
    eligible = clean.loc[scope].sort_values(sort_columns, kind="stable")
    if eligible.empty:
        raise ValueError("No eligible records.")

    if duplicate_kind not in ("exact", "conflicting", "mixed", "random"):
        raise ValueError("Unrecognized duplicate kind.")
    if not 0 < conflict_share < 1:
        raise ValueError("Conflict share must lie between 0 and 1.")
    kind = rng.choice(("exact", "conflicting", "mixed")) if duplicate_kind == "random" else duplicate_kind
    selected, _ = select_positions(
        len(eligible), rng, method=placement, rate=duplicate_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    count = len(selected)
    if kind == "mixed" and count < 2:
        raise ValueError("Mixed duplicate kinds require at least two copies.")

    copies = eligible.iloc[selected].copy().reset_index(drop=True)
    conflicts = (0 if kind == "exact" else count if kind == "conflicting" else
                 max(1, min(count - 1, round(count * conflict_share))))
    if conflicts:
        if conflict_column not in copies or conflict_column in key_columns or conflict_column == time_column:
            raise ValueError("Conflict column must exist and cannot be a key or time column.")
        if not pd.api.types.is_numeric_dtype(copies[conflict_column]):
            raise ValueError("This version requires a numeric conflict column.")
        if copies[conflict_column].isna().any() or conflict_offset == 0:
            raise ValueError("Conflicting values need nonmissing numbers and a nonzero offset.")
        if pd.api.types.is_integer_dtype(copies[conflict_column]):
            copies[conflict_column] = copies[conflict_column].astype(float)
        change = rng.sample(range(count), k=conflicts)
        copies.loc[change, conflict_column] += conflict_offset

    dirty = (pd.concat([clean, copies], ignore_index=True)
             .sort_values(sort_columns, kind="stable")
             .reset_index(drop=True))
    if len(dirty) != len(clean) + count or dirty.duplicated(subset=key_columns).sum() != count:
        raise AssertionError("Duplicate count differs from the selected count.")
    return dirty


In [ ]:
# Small business-like trial with composite keys and irregular transactions.
trial_time = pd.date_range("2025-08-01", periods=28, freq="11h", tz="UTC")
orders = pd.DataFrame({
    "time": trial_time.repeat(2),
    "branch": ["North", "South"] * len(trial_time),
    "amount": [float(i * 5 + 20) for i in range(len(trial_time) * 2)],
})
trial_dirty = add_duplicates(
    orders, Random(SEED), time_column="time", timezone="UTC",
    start_date="2025-08-01", end_date="2025-08-15",
    key_columns=["time", "branch"], conflict_column="amount", conflict_offset=5,
    duplicate_kind="mixed", placement="mixed", duplicate_rate=(0.20, 0.20),
    n_groups=2, grouped_share=0.70, conflict_share=0.50,
)
assert len(trial_dirty) > len(orders)
assert trial_dirty.duplicated(subset=["time", "branch"]).sum() == len(trial_dirty) - len(orders)
trial_dirty.loc[trial_dirty.duplicated(subset=["time", "branch"], keep=False)].head(12)


In [ ]:
# Apply to a freshly prepared CSV. This overwrites weather.csv.
clean = pd.read_csv(DATA_PATH)
rng = Random(SEED)
dirty = add_duplicates(
    clean, rng, time_column=TIME_COLUMN, timezone=TIMEZONE,
    start_date=START_DATE, end_date=END_DATE,
    key_columns=KEY_COLUMNS, conflict_column=DUPLICATE_CONFLICT_COLUMN,
    conflict_offset=DUPLICATE_CONFLICT_OFFSET, duplicate_kind=DUPLICATE_KIND,
    placement=DUPLICATE_PLACEMENT, duplicate_rate=DUPLICATE_RATE,
    n_groups=DUPLICATE_GROUPS, grouped_share=DUPLICATE_GROUPED_SHARE,
    conflict_share=DUPLICATE_CONFLICT_SHARE,
)
dirty.to_csv(DATA_PATH, index=False)
saved = pd.read_csv(DATA_PATH)


In [ ]:
saved.duplicated().any(), saved.duplicated(subset=KEY_COLUMNS).any()

In [ ]:
saved.duplicated().sum(), saved.duplicated(subset=KEY_COLUMNS).sum()

In [ ]:
saved.loc[saved.duplicated(subset=KEY_COLUMNS, keep=False)].head(20)

In [ ]:
# Verify that extra records were added and original records survive unchanged.
extra = saved.duplicated(subset=KEY_COLUMNS)
assert extra.sum() == len(saved) - len(clean)
assert pd.to_datetime(saved[TIME_COLUMN], utc=True).is_monotonic_increasing
originals = saved.loc[~extra].reset_index(drop=True)
pd.testing.assert_frame_equal(originals, clean.reset_index(drop=True), check_dtype=False, check_exact=False, rtol=1e-12)
print("Extra records:", int(extra.sum()))
print("Exact duplicate records:", int(saved.duplicated().sum()))
print("Conflicting-key copies:", int(extra.sum() - saved.duplicated().sum()))

local_dates = pd.to_datetime(clean[TIME_COLUMN], utc=True).dt.tz_convert(TIMEZONE).dt.date
saved_dates = pd.to_datetime(saved[TIME_COLUMN], utc=True).dt.tz_convert(TIMEZONE).dt.date
comparison = pd.DataFrame({"clean_rows": clean.groupby(local_dates).size(), "dirty_rows": saved.groupby(saved_dates).size()})
comparison.loc[comparison["clean_rows"].ne(comparison["dirty_rows"])].head(30)


In [ ]:
import matplotlib.pyplot as plt

# Extra records by local date and kind.
extra = saved.duplicated(subset=KEY_COLUMNS)
view = saved.loc[extra].copy()
view["date"] = pd.to_datetime(view[TIME_COLUMN], utc=True).dt.tz_convert(TIMEZONE).dt.date
view["kind"] = saved.duplicated().loc[view.index].map({True: "exact", False: "conflicting"})
local_dates = pd.to_datetime(clean[TIME_COLUMN], utc=True).dt.tz_convert(TIMEZONE).dt.date
all_dates = pd.Index(local_dates.unique(), name="date")
counts = (view.groupby(["date", "kind"]).size().unstack(fill_value=0)
          .reindex(all_dates, fill_value=0))
fig, ax = plt.subplots(figsize=(11, 3.5))
counts.plot.bar(ax=ax, stacked=True, width=0.9)
ax.set_title("Introduced duplicates by local date")
ax.set_xlabel("Date")
ax.set_ylabel("Extra records")
step = max(1, len(counts) // 10)
ax.set_xticks(range(0, len(counts), step), [str(d) for d in counts.index[::step]], rotation=45, ha="right")
plt.tight_layout()
plt.show()
